# CNMP — Carga silver (Resolução 277)

Lê o bronze (Lakehouse `mp_bronze`) e recarrega as tabelas normalizadas no
Warehouse `mp_silver`, usando o código de `python/src/modulos/cnmp/etl/`
do repositório `mpsp-jurimetria/proj202607`.

**Pré-requisitos:**
- O repositório precisa estar acessível para `pip install` (hoje está público).
- A extração bronze (`uv run python -m src.modulos.cnmp.etl.extract_bronze`,
  rodada localmente) já precisa ter gravado os JSONs em `mp_bronze`.
- Este notebook usa a identidade nativa do Fabric para autenticar no Lakehouse
  e no Warehouse — não precisa de Service Principal nem segredo nenhum aqui.
  É necessário que essa identidade tenha permissão de leitura no `mp_bronze`
  e de escrita no `mp_silver`.

In [ ]:
%pip install --quiet git+https://github.com/mpsp-jurimetria/proj202607.git#subdirectory=python

## Configuração

Preencha com os IDs do seu workspace/lakehouse/warehouse (Configurações do item
no portal Fabric → "Detalhes da conexão"). Não são segredos, mas evite deixar
valores reais commitados neste notebook num repositório público — prefira
rodar esta célula com os valores reais só na sessão do notebook, sem salvar.

In [ ]:
import os

os.environ["FABRIC_WORKSPACE_ID"] = "<id do workspace>"
os.environ["FABRIC_LAKEHOUSE_ID"] = "<id do lakehouse mp_bronze>"
os.environ["FABRIC_WAREHOUSE_SILVER_HOST"] = "<host>.datawarehouse.fabric.microsoft.com"
os.environ["FABRIC_WAREHOUSE_SILVER_NAME"] = "mp_silver"

In [ ]:
from src.infra.warehouse import get_silver_engine
from src.modulos.cnmp.etl.load_silver import carregar_silver

engine = get_silver_engine()
carregar_silver(engine, ambientes_ids=[282, 462])

## Verificação

In [ ]:
from sqlalchemy import text

tabelas = [
    "dim_ambiente", "dim_formulario", "dim_formulario_tipo_entidade",
    "dim_secao", "dim_campo", "dim_campo_opcao", "dim_campo_dependencia",
    "dim_entidade", "fato_instancia", "fato_resposta",
]

with engine.connect() as conn:
    for tabela in tabelas:
        total = conn.execute(text(f"SELECT COUNT(*) FROM {tabela}")).scalar()
        print(f"{tabela}: {total} linhas")